# Tokenizer demo: text → token ids → text

(Notebook version of `scripts/tokenize_demo.py`.)

A language model never sees letters. A **tokenizer** first chops text into *tokens* (whole words or word
pieces) and maps each to an integer id. The model works on those ids, and the tokenizer converts ids back to
text at the end. We use SmolLM2's tokenizer (byte-level BPE, 49,152 tokens) so that our model can later load
the published SmolLM2 weights.

Select the kernel **smalllm (.venv)** before running.

In [1]:
from huggingface_hub import hf_hub_download
from tokenizers import Tokenizer

REPO = "HuggingFaceTB/SmolLM2-135M"

C:\projects\smallllm\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load the tokenizer
`tokenizer.json` holds the vocabulary and the BPE merge rules. It is downloaded on the first run (small),
then reused from the cache in `~/.cache/huggingface`.

In [2]:
tok = Tokenizer.from_file(hf_hub_download(REPO, "tokenizer.json"))
print("vocab size:", tok.get_vocab_size())

vocab size: 49152


## 2. Encode
Edit `SENTENCE` and re-run to try your own text.

In [3]:
SENTENCE = "Once upon a time"

ids = tok.encode(SENTENCE).ids
print(f"text: {SENTENCE!r}")
print(f"ids:  {ids}")

text: 'Once upon a time'
ids:  [6403, 1980, 253, 655]


## 3. What each id stands for
`Ġ` is how the tokenizer writes a leading space, so `Ġupon` means " upon". Spaces belong to the *following*
word, which is why the first word has no `Ġ`.

In [4]:
for token_id in ids:
    print(f"{token_id:>6} -> {tok.id_to_token(token_id)!r}")

  6403 -> 'Once'
  1980 -> 'Ġupon'
   253 -> 'Ġa'
   655 -> 'Ġtime'


## 4. Decode and round trip
Decoding the ids must reproduce the original text exactly.

In [5]:
text = tok.decode(ids)
print(repr(text))
assert text == SENTENCE, "round trip should reproduce the original text"

'Once upon a time'


## 5. Try it: rare words split into pieces
Common words are a single token; rare words and names break into several. Change the list and look.

In [6]:
for word in ["hello", "unbelievably", "Pemberley", "supercalifragilistic", "12345"]:
    enc = tok.encode(word)
    print(f"{word!r:26} -> {len(enc.ids)} token(s): {enc.tokens}")

'hello'                    -> 1 token(s): ['hello']
'unbelievably'             -> 4 token(s): ['un', 'belie', 'v', 'ably']
'Pemberley'                -> 3 token(s): ['P', 'ember', 'ley']
'supercalifragilistic'     -> 6 token(s): ['super', 'cal', 'if', 'rag', 'il', 'istic']
'12345'                    -> 5 token(s): ['1', '2', '3', '4', '5']
